# 파이썬 머신러닝 완벽가이드 8.4장 복습과제  
## 텍스트 분류 실습: 20 뉴스그룹 분류


## 1. 실습 개요

- 텍스트 분류는 문서의 내용을 보고 해당 문서가 어떤 범주에 속하는지 예측하는 지도학습 문제
- 이번 실습에서는 사이킷런이 제공하는 `fetch_20newsgroups()` 데이터 사용
- 이 데이터는 20개의 뉴스그룹 주제로 구성되어 있으며, 각 문서는 특정 주제의 뉴스그룹 글이다.

텍스트 분류의 기본 흐름

1. 데이터 로딩
2. 텍스트 정규화 및 불필요 정보 제거
3. 텍스트 피처 벡터화
4. 머신러닝 모델 학습
5. 테스트 데이터 예측
6. 정확도 평가
7. Pipeline과 GridSearchCV를 이용한 튜닝

## 2. 관련 이론 정리

### 2.1 텍스트 데이터와 피처 벡터화

머신러닝 모델은 일반적으로 숫자형 데이터를 입력으로 받는다. 따라서 문장이나 문서처럼 문자로 이루어진 데이터는 바로 모델에 넣을 수 없다. 텍스트를 단어 단위의 숫자형 벡터로 바꾸는 과정을 **피처 벡터화**라고 한다.

대표적인 벡터화 방식은 다음과 같다.

- **Count Vectorization**: 문서에 특정 단어가 몇 번 등장했는지를 기반으로 벡터화한다.
- **TF-IDF Vectorization**: 특정 문서 안에서는 자주 등장하지만 전체 문서에서는 너무 흔하지 않은 단어에 더 높은 가중치를 준다.

Count 방식은 단순하고 직관적이지만, 모든 문서에서 자주 등장하는 일반적인 단어까지 중요하게 볼 수 있다. 반면 TF-IDF는 이런 문제를 완화해 텍스트 분류에서 더 좋은 성능을 보이는 경우가 많다.

### 2.2 20 뉴스그룹 데이터의 특징

`fetch_20newsgroups()` 데이터에는 뉴스 본문뿐 아니라 제목, 작성자, 이메일, 소속, 인용문 같은 정보가 포함될 수 있다. 그런데 이런 정보는 실제 문서 내용보다 특정 주제를 쉽게 암시할 수 있다. 예를 들어 이메일 주소나 뉴스그룹 헤더가 분류 정답과 강하게 연결되어 있으면 모델이 본문 의미를 학습했다기보다 부가 정보를 외운 것처럼 높은 성능을 낼 수 있다.

따라서 이번 실습에서는 `remove=('headers', 'footers', 'quotes')` 옵션을 사용해 헤더, 푸터, 인용문을 제거하고 본문 중심으로 분류를 수행한다.

### 2.3 Logistic Regression을 사용하는 이유

텍스트를 BOW 또는 TF-IDF 방식으로 벡터화하면 대부분의 값이 0인 **희소 행렬**이 만들어진다. 이런 희소 행렬 기반의 텍스트 분류에는 로지스틱 회귀, 선형 SVM, 나이브 베이즈 등이 자주 사용된다.

이번 실습에서는 교재 흐름에 맞춰 **Logistic Regression**을 사용한다. 로지스틱 회귀는 이름에 회귀가 들어가지만, 분류 문제에도 널리 사용되는 선형 분류 모델이다.

In [1]:
# 기본 라이브러리 불러오기
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

## 3. 데이터 로딩 및 구조 확인

먼저 전체 20 뉴스그룹 데이터를 불러와 데이터가 어떤 key를 가지고 있는지, target 클래스는 어떻게 구성되어 있는지 확인한다.

In [2]:
# 20 뉴스그룹 데이터 로딩
# subset='all'은 train/test를 모두 포함한 전체 데이터를 의미한다.
news_data = fetch_20newsgroups(subset='all', random_state=156)

print(news_data.keys())

dict_keys(['data', 'filenames', 'target_names', 'target', 'DESCR'])


In [3]:
# target 클래스 분포와 클래스 이름 확인
print('target 클래스의 값과 분포도')
print(pd.Series(news_data.target).value_counts().sort_index())

print('target 클래스의 이름들')
print(news_data.target_names)

target 클래스의 값과 분포도
0     799
1     973
2     985
3     982
4     963
5     988
6     975
7     990
8     996
9     994
10    999
11    991
12    984
13    990
14    987
15    997
16    910
17    940
18    775
19    628
Name: count, dtype: int64
target 클래스의 이름들
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [4]:
# 첫 번째 문서 내용 확인
print(news_data.data[0][:2000])

From: egreen@east.sun.com (Ed Green - Pixel Cruncher)
Subject: Re: Observation re: helmets
Organization: Sun Microsystems, RTP, NC
Lines: 21
Distribution: world
Reply-To: egreen@east.sun.com
NNTP-Posting-Host: laser.east.sun.com

In article 211353@mavenry.altcit.eskimo.com, maven@mavenry.altcit.eskimo.com (Norman Hamer) writes:
> 
> The question for the day is re: passenger helmets, if you don't know for 
>certain who's gonna ride with you (like say you meet them at a .... church 
>meeting, yeah, that's the ticket)... What are some guidelines? Should I just 
>pick up another shoei in my size to have a backup helmet (XL), or should I 
>maybe get an inexpensive one of a smaller size to accomodate my likely 
>passenger? 

If your primary concern is protecting the passenger in the event of a
crash, have him or her fitted for a helmet that is their size.  If your
primary concern is complying with stupid helmet laws, carry a real big
spare (you can put a big or small head in a big helmet, bu

## 4. 텍스트 정규화: 헤더, 푸터, 인용문 제거

뉴스그룹 원문에는 본문 외에도 헤더, 푸터, 인용문이 들어 있다. 이 정보들은 분류 성능을 비정상적으로 높일 수 있으므로 제거한다.  
또한 `subset='train'`, `subset='test'`를 사용해 학습 데이터와 테스트 데이터를 분리해서 가져온다.

In [5]:
# 학습 데이터 로딩: headers, footers, quotes 제거
train_news = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes'),
    random_state=156
)

X_train = train_news.data
y_train = train_news.target

# 테스트 데이터 로딩: 동일하게 headers, footers, quotes 제거
test_news = fetch_20newsgroups(
    subset='test',
    remove=('headers', 'footers', 'quotes'),
    random_state=156
)

X_test = test_news.data
y_test = test_news.target

print('학습 데이터 크기:', len(X_train))
print('테스트 데이터 크기:', len(X_test))

학습 데이터 크기: 11314
테스트 데이터 크기: 7532


## 5. CountVectorizer 기반 피처 벡터화와 모델 학습

먼저 CountVectorizer를 사용해 텍스트를 단어 빈도 기반 벡터로 변환한다.  
주의할 점은 **테스트 데이터에는 fit_transform()을 사용하면 안 된다**는 것이다. 학습 데이터로 fit된 벡터화 객체를 이용해 테스트 데이터는 transform만 해야 학습 데이터와 테스트 데이터의 피처 구조가 동일하게 유지된다.

In [6]:
# CountVectorizer로 학습 데이터 벡터화
cnt_vect = CountVectorizer()
cnt_vect.fit(X_train)

X_train_cnt_vect = cnt_vect.transform(X_train)
X_test_cnt_vect = cnt_vect.transform(X_test)

print('학습 데이터 CountVectorizer Shape:', X_train_cnt_vect.shape)
print('테스트 데이터 CountVectorizer Shape:', X_test_cnt_vect.shape)

학습 데이터 CountVectorizer Shape: (11314, 101631)
테스트 데이터 CountVectorizer Shape: (7532, 101631)


In [10]:
# CountVectorizer + Logistic Regression 모델 학습/예측/평가
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_cnt_vect, y_train)

pred = lr_clf.predict(X_test_cnt_vect)
print('CountVectorized Logistic Regression의 예측 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

CountVectorized Logistic Regression의 예측 정확도: 0.617


## 6. TF-IDF 기반 피처 벡터화와 모델 학습

이번에는 Count 기반 대신 TF-IDF 기반으로 벡터화한다. TF-IDF는 개별 문서에서 자주 등장하지만 전체 문서에서 지나치게 흔하지 않은 단어를 더 중요하게 반영한다.

In [7]:
# TF-IDF 벡터화
# 주의: 학습 데이터로 fit한 뒤, 테스트 데이터는 transform만 수행한다.
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(X_train)

X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

print('학습 데이터 TfidfVectorizer Shape:', X_train_tfidf_vect.shape)
print('테스트 데이터 TfidfVectorizer Shape:', X_test_tfidf_vect.shape)

학습 데이터 TfidfVectorizer Shape: (11314, 101631)
테스트 데이터 TfidfVectorizer Shape: (7532, 101631)


In [8]:
# TF-IDF + Logistic Regression 모델 학습/예측/평가
lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)

pred = lr_clf.predict(X_test_tfidf_vect)
print('TF-IDF Logistic Regression의 예측 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

TF-IDF Logistic Regression의 예측 정확도: 0.678


## 7. TF-IDF 파라미터 조정

텍스트 분류 성능은 벡터화 방식과 전처리 파라미터에 영향을 많이 받는다. 여기서는 다음 옵션을 적용한다.

- `stop_words='english'`: 영어 불용어 제거
- `ngram_range=(1, 2)`: 단일 단어뿐 아니라 연속된 두 단어 조합까지 피처로 사용
- `max_df=300`: 너무 많은 문서에서 등장하는 단어를 제외

In [9]:
# TF-IDF 파라미터 조정
# stop_words: 영어 불용어 제거
# ngram_range: unigram + bigram 사용
# max_df: 너무 자주 등장하는 단어 제외
tfidf_vect = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_df=300
)

tfidf_vect.fit(X_train)
X_train_tfidf_vect = tfidf_vect.transform(X_train)
X_test_tfidf_vect = tfidf_vect.transform(X_test)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit(X_train_tfidf_vect, y_train)

pred = lr_clf.predict(X_test_tfidf_vect)
print('파라미터 조정 TF-IDF + Logistic Regression의 예측 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

파라미터 조정 TF-IDF + Logistic Regression의 예측 정확도: 0.690


## 8. GridSearchCV로 Logistic Regression 하이퍼파라미터 튜닝

이번에는 로지스틱 회귀의 규제 강도를 조절하는 `C` 값을 바꿔가며 최적의 하이퍼파라미터를 찾는다.  
`C` 값이 클수록 규제가 약해지고, 작을수록 규제가 강해진다.

In [11]:
# GridSearchCV로 Logistic Regression의 C 값 튜닝
params = {'C': [0.01, 0.1, 1, 5, 10]}

grid_cv_lr = GridSearchCV(
    lr_clf,
    param_grid=params,
    cv=3,
    scoring='accuracy',
    verbose=1
)

grid_cv_lr.fit(X_train_tfidf_vect, y_train)

print('Logistic Regression best C parameter:', grid_cv_lr.best_params_)
print('교차검증 최고 정확도:', grid_cv_lr.best_score_)

pred = grid_cv_lr.predict(X_test_tfidf_vect)
print('GridSearchCV 적용 후 테스트 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Logistic Regression best C parameter: {'C': 10}
교차검증 최고 정확도: 0.7519009137377873
GridSearchCV 적용 후 테스트 정확도: 0.704


## 9. Pipeline 사용

Pipeline을 사용하면 텍스트 벡터화와 모델 학습 과정을 하나의 흐름으로 묶을 수 있다.  
즉, 별도로 `TfidfVectorizer.fit()`, `transform()`, `LogisticRegression.fit()`을 나누어 작성하지 않아도 된다.

In [12]:
# TfidfVectorizer와 LogisticRegression을 Pipeline으로 연결
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_df=300
    )),
    ('lr_clf', LogisticRegression(solver='liblinear', C=10))
])

# Pipeline의 fit과 predict만으로 벡터화 + 학습 + 예측 수행
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print('Pipeline을 통한 Logistic Regression의 예측 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

Pipeline을 통한 Logistic Regression의 예측 정확도: 0.704


## 10. Pipeline과 GridSearchCV의 결합

Pipeline을 GridSearchCV에 넣으면 벡터화 파라미터와 모델 하이퍼파라미터를 한 번에 튜닝할 수 있다.  
이때 파라미터 이름은 `객체이름__파라미터명` 형식으로 작성한다. 예를 들어 `tfidf_vect__ngram_range`는 Pipeline 안의 `tfidf_vect` 단계에 있는 `ngram_range` 파라미터를 의미한다.

주의: 전체 조합을 크게 잡으면 실행 시간이 오래 걸릴 수 있다. 아래 코드는 교재 흐름을 따르되, 과제 실행 편의를 위해 `n_jobs=-1`을 추가함

In [13]:
# Pipeline 생성
pipeline = Pipeline([
    ('tfidf_vect', TfidfVectorizer(stop_words='english')),
    ('lr_clf', LogisticRegression(solver='liblinear'))
])

# Pipeline 내부 객체의 파라미터를 GridSearchCV에서 튜닝
params = {
    'tfidf_vect__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf_vect__max_df': [100, 300, 700],
    'lr_clf__C': [1, 5, 10]
}

grid_cv_pipe = GridSearchCV(
    pipeline,
    param_grid=params,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_cv_pipe.fit(X_train, y_train)

print('Best parameters:', grid_cv_pipe.best_params_)
print('Best CV score:', grid_cv_pipe.best_score_)

pred = grid_cv_pipe.predict(X_test)
print('Pipeline + GridSearchCV 테스트 정확도: {0:.3f}'.format(
    accuracy_score(y_test, pred)
))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best parameters: {'lr_clf__C': 10, 'tfidf_vect__max_df': 700, 'tfidf_vect__ngram_range': (1, 2)}
Best CV score: 0.7550828826229531
Pipeline + GridSearchCV 테스트 정확도: 0.702


## 11. 분류 결과 상세 확인

정확도만 보면 전체 성능은 알 수 있지만, 어떤 클래스에서 잘 맞추고 어떤 클래스에서 약한지는 알기 어렵다.  
`classification_report`를 사용하면 클래스별 precision, recall, f1-score를 확인할 수 있다.

In [14]:
# 최종 모델 기준으로 상세 분류 리포트 확인
# target_names를 함께 넣으면 숫자 라벨 대신 뉴스그룹 이름을 볼 수 있다.
final_pred = grid_cv_pipe.predict(X_test)

print(classification_report(
    y_test,
    final_pred,
    target_names=test_news.target_names
))

                          precision    recall  f1-score   support

             alt.atheism       0.54      0.46      0.50       319
           comp.graphics       0.63      0.71      0.67       389
 comp.os.ms-windows.misc       0.67      0.65      0.66       394
comp.sys.ibm.pc.hardware       0.68      0.67      0.68       392
   comp.sys.mac.hardware       0.74      0.68      0.71       385
          comp.windows.x       0.84      0.71      0.77       395
            misc.forsale       0.72      0.79      0.76       390
               rec.autos       0.50      0.79      0.61       396
         rec.motorcycles       0.81      0.77      0.79       398
      rec.sport.baseball       0.83      0.81      0.82       397
        rec.sport.hockey       0.90      0.89      0.89       399
               sci.crypt       0.87      0.70      0.77       396
         sci.electronics       0.60      0.60      0.60       393
                 sci.med       0.79      0.80      0.80       396
         

## 12. 실습 결과 정리

1. 텍스트 데이터는 바로 머신러닝 모델에 넣을 수 없기 때문에 피처 벡터화가 필요하다.
2. CountVectorizer는 단어 등장 횟수를 기반으로 단순하게 벡터화한다.
3. TfidfVectorizer는 문서 내 빈도와 전체 문서에서의 희소성을 함께 반영해 단어의 중요도를 계산한다.
4. 테스트 데이터는 학습 데이터로 fit된 벡터화 객체를 이용해 transform해야 한다.
5. 텍스트 벡터화 결과는 대부분 희소 행렬 형태가 된다.
6. 희소 행렬 기반 텍스트 분류에는 로지스틱 회귀, 선형 SVM, 나이브 베이즈 등이 자주 사용된다.
7. Pipeline을 사용하면 벡터화와 모델 학습을 하나의 과정으로 묶을 수 있다.
8. Pipeline과 GridSearchCV를 결합하면 벡터화 파라미터와 모델 하이퍼파라미터를 함께 튜닝할 수 있다.

실습 흐름상 일반적으로 CountVectorizer보다 TF-IDF 기반 모델의 성능이 더 높게 나타나며, 불용어 제거, n-gram, max_df, C 값 조정 등을 통해 성능을 조금 더 개선할 수 있다.

## 13. 추가 복습 메모

- `fit_transform()`은 학습 데이터에 사용한다.
- 테스트 데이터에는 `fit_transform()`이 아니라 `transform()`을 사용한다.
- `remove=('headers', 'footers', 'quotes')`는 본문 외 정보를 제거하는 옵션이다.
- `ngram_range=(1, 2)`는 단어 1개와 연속된 단어 2개 조합을 모두 피처로 사용한다.
- `max_df`는 너무 많은 문서에 등장하는 단어를 제외하는 기준이다.
- `Pipeline` 내부 파라미터를 GridSearchCV에서 지정할 때는 `단계명__파라미터명` 형식으로 쓴다.